# Audit exploratoire — Amazon Reviews 2023

## Pourquoi ce notebook existe

Ce notebook est le **compagnon interactif** de la chaîne d'audit `src/data/audit/01..04_*.py`. Sa raison d'être :

- Les 4 scripts produisent du **JSON** + **PNG** + un **rapport markdown** committables et reproductibles. C'est la source de vérité.
- Le notebook permet, en parallèle, d'**explorer librement** le dataset (tester une nouvelle requête polars, regarder un échantillon de lignes, vérifier visuellement une distribution surprenante) sans casser le pipeline officiel.
- Toute observation d'intérêt qui sort du notebook **doit être remontée dans le rapport** `reports/audit_v1_amazon_reviews_2023.md` (le notebook n'est pas le livrable, le rapport l'est).

## Comment on explore un périmètre lourd sans tomber dans le biais d'exploration

Le périmètre actif (cf. `BRAIN/decisions.md` D-008) couvre **plusieurs centaines de millions de lignes**. Deux pièges symétriques à éviter :

| Piège | Pourquoi c'est mauvais |
|---|---|
| **Charger 1 seule catégorie** | C'est aterrir sur 10 % de la planète et conclure sur 100 %. On reproduit R18 (biais de supposition explicite) qu'on a justement combattu en méta-first |
| **Streamer le full** | Cellules à 1-5 min chacune, l'interactivité disparaît, on n'explore plus rien |

**La réponse** : un **sample stratifié représentatif** sur l'ensemble du périmètre. Concrètement, on tire `N` lignes par catégorie via un **Bernoulli step uniforme dans le fichier** (1 ligne sur K, K = `n_total // N`). Chaque catégorie est échantillonnée **indépendamment** — donc Appliances n'est pas écrasée par Home_and_Kitchen, et l'échantillon de chaque cat est **uniformément distribué dans le fichier** (pas concentré sur le début ou la fin).

Avantages :
- Tient en RAM (~1-2 M lignes total → quelques centaines de MB en pandas)
- Cellules redeviennent interactives (< 5 s)
- Tu vois la planète entière à basse résolution, pas un patch
- Reproductible : `seed=42` partout

Limite assumée :
- Les chiffres calculés ici sont **descriptifs sur l'échantillon**, pas canoniques. Pour les chiffres de production, voir `reports/audit_v1_amazon_reviews_2023.md` (issu du full streaming).

## Place dans le projet

```
Phase P02 — Données comme matière première
└── Sous-todo 1.1 — Audit dataset brut  (← TU ES ICI)
    ├── notebooks/00_meta_exploration.ipynb   [méta-first : Range probes + filtre périmètre]
    ├── notebooks/01_audit_exploratoire.ipynb [ce fichier — sample stratifié interactif]
    ├── src/data/audit/01..04                 [code Python officiel — full streaming]
    ├── reports/audit_v1_*.md                 [synthèse humaine canonique]
    ├── reports/audit_*.json                  [métriques machine-readable du full]
    └── reports/figures/*.png                 [graphes officiels du full]

→ Sous-todo 1.2 — Cleaning + feature engineering + split (à venir, C06)
→ Sous-todo 1.3 — Anti-leakage : 5 patterns (à venir, C07)
→ Sous-todo 1.4 — Validation Pandera (à venir, C08)
```

## Discipline (rappel des invariants C05)

1. **Audit décrit, cleaning corrige.** On ne modifie **pas** les fichiers `data/raw/full/` depuis ce notebook. Toute correction part en C06.
2. **Pas de chiffre sans source.** Les chiffres canoniques cités dans le rapport doivent venir d'un JSON métriques généré par `make audit` sur le full, pas des cellules de ce notebook (qui travaillent sur sample).
3. **Reproductibilité.** `seed=42` partout, sample stratifié déterministe, plus aucun chiffre dépendant de l'ordre des lignes.

## Périmètre actif

Les catégories retenues vivent dans `src/data/audit/__init__.py` (single source of truth — D-008). Si tu modifies cette liste, **toute la chaîne** (scripts 01..04 + ce notebook) se réaligne automatiquement au prochain run.

## Comment exécuter

Préalable : `make install-data` puis `make audit-load` doivent avoir tourné (les parquets doivent exister dans `data/raw/full/{reviews,meta}/`).

Puis ouvrir ce notebook avec `jupyter lab notebooks/` ou via VSCode (extension Jupyter), et exécuter cellule par cellule.

---

## Setup — imports et chemins

On importe :
- **polars** comme moteur lazy/streaming (pour le scan + sample stratifié sans matérialiser le fichier complet)
- **pandas** pour l'affichage Jupyter (jolies tables, `.describe()`, `.plot()`)
- **matplotlib + seaborn** pour les graphiques
- **`CATEGORIES` et `COMPLEX_COLS_TO_DROP`** depuis `src.data.audit` (single source of truth — D-008)

Le `REPO_ROOT` se résout automatiquement qu'on lance le notebook depuis `notebooks/` ou depuis la racine du repo.

In [ ]:
import sys
from pathlib import Path

import polars as pl
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

REPO_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
REVIEWS_DIR = REPO_ROOT / 'data' / 'raw' / 'full' / 'reviews'
META_DIR = REPO_ROOT / 'data' / 'raw' / 'full' / 'meta'
REPORTS_DIR = REPO_ROOT / 'reports'

# Permettre l'import du package src/ quand on lance depuis notebooks/
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.data.audit import CATEGORIES, COMPLEX_COLS_TO_DROP

sns.set_theme(style='whitegrid', context='notebook')

assert REVIEWS_DIR.exists() and any(REVIEWS_DIR.iterdir()), (
    'Les parquets reviews ne sont pas téléchargés. Lance `make audit-load` depuis la racine.'
)

print(f'Repo root            : {REPO_ROOT}')
print(f'Périmètre actif      : {len(CATEGORIES)} catégories (D-008)')
print(f'Catégories           : {CATEGORIES}')
print(f'Colonnes à drop      : {COMPLEX_COLS_TO_DROP}')
print(f'Reviews parquet (n)  : {sum(1 for _ in REVIEWS_DIR.glob("*.parquet"))}')
print(f'Meta parquet (n)     : {sum(1 for _ in META_DIR.glob("*.parquet"))}')

---

## Section 1 — Le sample stratifié représentatif

**Ce qu'on fait** : pour chaque catégorie du périmètre, prélever `N_PER_CAT` lignes uniformément distribuées dans le fichier via un **Bernoulli step**. Les fichiers sont scannés en lazy, on ne charge en RAM que les lignes effectivement gardées.

**Pourquoi pas un sample aléatoire global ?** Parce qu'il serait écrasé par les grosses catégories : sur 348 M reviews dont 67 M dans Home_and_Kitchen et 2 M dans Appliances, un échantillon aléatoire global de 1 M lignes contiendrait ~190 K Home_and_Kitchen et ~6 K Appliances — Appliances serait sous-représentée à l'audit. Avec la stratification, **chaque cat est échantillonnée à hauteur de `N_PER_CAT`**, donc on a la même finesse partout.

**Pourquoi un step et pas `lf.sample(n, seed)` ?** Parce que `sample()` matérialise le LazyFrame complet d'abord, donc il faudrait avoir les 67 M lignes de Home_and_Kitchen en RAM avant de pouvoir échantillonner — défait l'intérêt. Le pattern `filter(pl.int_range(pl.len()) % step == 0)` reste lazy : polars lit séquentiellement, garde 1 ligne sur K, ne matérialise que ce qui passe le filtre.

**Choix de N_PER_CAT** :
- Petit (10 K) : très rapide, peu de précision pour les distributions queue-longue
- Moyen (100 K) : équilibre raisonnable pour un audit interactif (~1,5 M total sur 15 cat ≈ 200-400 MB RAM)
- Grand (1 M) : précision proche du full sur les agrégats simples, mais plus lent à charger

On démarre à **100 000 par cat**, ajustable via la constante en haut de cellule.

In [ ]:
N_PER_CAT = 100_000  # lignes par catégorie. Total ≈ N_PER_CAT × len(CATEGORIES).
SEED = 42


def stratified_sample(dir_path: Path, n_per_cat: int = N_PER_CAT, seed: int = SEED) -> pl.DataFrame:
    """Sample stratifié représentatif sur les CATEGORIES du périmètre actif (D-008).

    Pour chaque cat, on scan le parquet, on drop les colonnes complexes
    (cf. COMPLEX_COLS_TO_DROP — types hétérogènes List[Struct]/String selon
    polars natif vs fallback pandas chunked à l'écriture), puis on garde
    1 ligne sur K via Bernoulli step pour avoir un échantillon uniforme dans
    le fichier. Le shuffle final déterministe (seed) homogénéise l'ordre.
    """
    parts: list[pl.DataFrame] = []
    for cat in CATEGORIES:
        path = dir_path / f'{cat}.parquet'
        lf = pl.scan_parquet(path)
        # Drop par nom les colonnes complexes hétérogènes (cohérent avec scan_concat du package)
        present = [c for c in COMPLEX_COLS_TO_DROP if c in lf.collect_schema().names()]
        if present:
            lf = lf.drop(present)
        # Compter les lignes via streaming engine (pas de matérialisation)
        n_total = lf.select(pl.len()).collect(engine='streaming').item()
        # Bernoulli step : 1 ligne sur K, où K = n_total // n_per_cat (clamp à 1 si petite cat)
        step = max(1, n_total // n_per_cat)
        df = (
            lf.with_row_index()
              .filter(pl.col('index') % step == 0)
              .drop('index')
              .with_columns(pl.lit(cat).alias('_source_category'))
              .collect(engine='streaming')
        )
        # Cap dur si le step a légèrement sur-fourni (effet d'arrondi entier)
        if df.height > n_per_cat:
            df = df.sample(n=n_per_cat, seed=seed)
        parts.append(df)
        print(f'  {cat:35s}  n_total={n_total:>12_}  step={step:>5}  sample={df.height:>7_}')
    return pl.concat(parts, how='diagonal_relaxed')


print(f'\n=== Sample stratifié reviews (N_PER_CAT={N_PER_CAT:_}) ===')
reviews_df = stratified_sample(REVIEWS_DIR)
print(f'\nTotal sample reviews : {reviews_df.height:_} lignes  ({reviews_df.estimated_size("mb"):.1f} MB)')
reviews_df.head(3)

In [ ]:
print(f'=== Sample stratifié meta (N_PER_CAT={N_PER_CAT:_}) ===')
meta_df = stratified_sample(META_DIR)
print(f'\nTotal sample meta : {meta_df.height:_} lignes  ({meta_df.estimated_size("mb"):.1f} MB)')
meta_df.head(3)

---

## Section 2 — Volumes : sample vs canonique

**Ce qu'on fait** : compter les lignes du sample par catégorie et **comparer** avec les chiffres canoniques du `reports/audit_distributions_metrics.json` (issu du full streaming).

**Ce qu'on cherche à valider** : que la stratification a bien fonctionné — chaque cat doit avoir ~`N_PER_CAT` lignes (sauf les très petites cat où on a tout pris). Un écart inattendu signalerait un bug dans la fonction `stratified_sample`.

In [ ]:
import json

# Comptes par cat sur le sample
sample_per_cat_reviews = (
    reviews_df.group_by('_source_category')
    .agg(pl.len().alias('n_sample'))
    .sort('n_sample', descending=True)
    .to_pandas().set_index('_source_category')
)

# Chiffres canoniques (full) si le rapport est dispo
metrics_path = REPORTS_DIR / 'audit_distributions_metrics.json'
if metrics_path.exists():
    metrics = json.loads(metrics_path.read_text(encoding='utf-8'))
    canon = pd.Series(metrics['reviews_per_cat'], name='n_full').sort_values(ascending=False)
    comparison = sample_per_cat_reviews.join(canon, how='outer')
    comparison['ratio_full_sur_sample'] = (comparison['n_full'] / comparison['n_sample']).round(0)
    print('Reviews par catégorie — sample stratifié vs full canonique :')
    display(comparison)
else:
    print('reports/audit_distributions_metrics.json absent (lance `make audit-dist`). Affichage du sample seulement.')
    display(sample_per_cat_reviews)

---

## Section 3 — Audit qualité sur sample (NaN, doublons)

**Ce qu'on fait** : recalculer NaN ratio et n_unique sur le sample. Comparer aux chiffres canoniques pour validation.

**Important** : sur le sample, les `n_unique_parent_asin` / `n_unique_user_id` seront **plus petits** que sur le full (par construction). C'est la **proportion** qui doit rester proche, pas la valeur absolue. Pour les chiffres absolus, lire `audit_qualite_metrics.json` ou le rapport.

**Ce qu'on cherche** :
- Confirmer que les colonnes reviews sont à 0 % NaN (dataset propre — devrait se voir dès le sample)
- Voir la concentration NaN sur metadata : `subtitle` ~85 %, `author` ~89 %, `price` ~58 % (mesurés sur le full D-008)
- Comparer le duplication factor `parent_asin` sample vs full

In [ ]:
# NaN ratio reviews — sur sample (~1,5 M lignes), pandas eager OK
nan_reviews = reviews_df.to_pandas().isna().mean().sort_values(ascending=False)
nan_reviews.to_frame('nan_ratio_sample').head(15)

In [ ]:
# NaN ratio meta — idem
nan_meta = meta_df.to_pandas().isna().mean().sort_values(ascending=False)
nan_meta.to_frame('nan_ratio_sample').head(20)

In [ ]:
# Unicité sur sample — proportions doivent matcher le full ; valeurs absolues non
uniqueness = reviews_df.select(
    n_total=pl.len(),
    n_unique_parent_asin=pl.col('parent_asin').n_unique(),
    n_unique_user_id=pl.col('user_id').n_unique(),
    n_unique_asin=pl.col('asin').n_unique(),
).to_dicts()[0]

duplication_factor = uniqueness['n_total'] / max(uniqueness['n_unique_parent_asin'], 1)
users_repeat_ratio = 1 - uniqueness['n_unique_user_id'] / uniqueness['n_total']

print(f"Sample stratifié — {uniqueness['n_total']:_} reviews")
print(f"  duplication factor parent_asin = {duplication_factor:.2f}  (full canonique : 13,21)")
print(f"  ratio users répétés            = {users_repeat_ratio*100:.1f} %  (full canonique : ~74 %)")
print()
print('Note : sur le sample stratifié les valeurs absolues sont nécessairement plus petites,')
print('mais les RATIOS (duplication factor, % users répétés) doivent rester proches du full.')

---

## Section 4 — Distributions visuelles

**Ce qu'on fait** : tracer rapidement les 5 distributions clés (catégorielle reviews, catégorielle meta, ratings, temporelle, prix). Les graphiques officiels (issus du full) sont déjà dans `reports/figures/`. Ici on est en mode **vérification visuelle rapide** sur sample : un pic suspect ou un trou bizarre nous alerte.

**Différence avec les scripts** : ici on plot dans le notebook (interactif), là-bas on sauve en PNG (commitable). Mêmes méthodes, livrables différents.

**Patterns défensifs réutilisés** (cf. acte 3 du cours C05) :
- `pl.coalesce` pour parser `timestamp` qui peut être Int64 ms ou String ISO selon que le parquet vient du sink polars natif ou du fallback pandas chunked
- Regex `[^\d.]` pour parser le champ `price` qui mélange chaînes (`"$12.99"`, `"None"`) et nombres

In [ ]:
# Distribution catégorielle — reviews + meta côte à côte
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

rev_counts = reviews_df['_source_category'].value_counts().sort('count', descending=True).to_pandas()
rev_counts.plot(kind='barh', x='_source_category', y='count', ax=axes[0],
                color='#2563eb', legend=False)
axes[0].set_title(f'Reviews — sample stratifié ({reviews_df.height:_} lignes)')
axes[0].invert_yaxis()

meta_counts = meta_df['_source_category'].value_counts().sort('count', descending=True).to_pandas()
meta_counts.plot(kind='barh', x='_source_category', y='count', ax=axes[1],
                 color='#16a34a', legend=False)
axes[1].set_title(f'Meta — sample stratifié ({meta_df.height:_} lignes)')
axes[1].invert_yaxis()

plt.tight_layout()
plt.show()

In [ ]:
# Distribution des ratings
ratings = (
    reviews_df.group_by(
        pl.col('rating').cast(pl.Float64, strict=False).round(0).cast(pl.Int64).alias('r')
    )
    .agg(pl.len().alias('n'))
    .sort('r')
)

fig, ax = plt.subplots(figsize=(7, 4))
ratings.to_pandas().set_index('r')['n'].plot(kind='bar', ax=ax, color='#f59e0b')
ax.set_title(f'Distribution ratings (sample stratifié, {reviews_df.height:_} reviews)')
ax.set_xlabel('Note (1-5)')
ax.set_ylabel('Effectif sample')
plt.tight_layout()
plt.show()

pct_5 = ratings.filter(pl.col('r') == 5)['n'].item() / reviews_df.height * 100
pct_1 = ratings.filter(pl.col('r') == 1)['n'].item() / reviews_df.height * 100
print(f'5★ sur sample : {pct_5:.1f} %  (full canonique : 65,5 %)')
print(f'1★ sur sample : {pct_1:.1f} %  (full canonique :  9,9 %)')

In [ ]:
# Distribution temporelle — pattern coalesce pour gérer timestamp Int64 ms vs String ISO
ts_str = pl.col('timestamp').cast(pl.Utf8, strict=False)
year_expr = pl.coalesce(
    ts_str.str.to_datetime(strict=False).dt.year(),
    ts_str.cast(pl.Int64, strict=False).cast(pl.Datetime(time_unit='ms'), strict=False).dt.year(),
).alias('year')

by_year = (
    reviews_df.with_columns(year_expr)
    .filter(pl.col('year').is_not_null())
    .group_by('year').agg(pl.len().alias('n'))
    .sort('year')
)

fig, ax = plt.subplots(figsize=(11, 4))
by_year.to_pandas().set_index('year')['n'].plot(kind='bar', ax=ax, color='#7c3aed')
ax.set_title(f'Distribution temporelle (sample stratifié, {reviews_df.height:_} reviews)')
ax.set_xlabel('Année')
ax.set_ylabel('Reviews / an (sample)')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# Distribution prix — parsing string + clip p99 + log scale
import numpy as np

prices = (
    meta_df.with_columns(
        pl.col('price')
        .cast(pl.Utf8, strict=False)
        .str.replace_all(r'[^\d.]', '')
        .cast(pl.Float64, strict=False)
        .alias('_price_num')
    )
    .filter(pl.col('_price_num').is_not_null() & (pl.col('_price_num') > 0))
    ['_price_num'].to_numpy()
)

if len(prices) > 0:
    p99 = float(np.quantile(prices, 0.99))
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].hist(np.clip(prices, 0, p99), bins=80, color='#16a34a')
    axes[0].set_title(f'Prix linéaire (clip p99={p99:.0f}$, n={len(prices):_})')
    axes[0].set_xlabel('Prix ($)')
    axes[1].hist(np.log1p(prices), bins=80, color='#16a34a')
    axes[1].set_title('Prix log(1+x)')
    axes[1].set_xlabel('log(1 + prix)')
    plt.tight_layout()
    plt.show()
    pct_below_50 = (prices < 50).mean() * 100
    print(f'Sample meta avec prix valide : {len(prices):_} sur {meta_df.height:_} ({len(prices)/meta_df.height*100:.1f} %)')
    print(f'%% prix < 50$ sur sample      : {pct_below_50:.1f} %  (full canonique : 82,0 %)')
    print(f'Médiane sample / canonique   : {np.median(prices):.2f}$ / 18,99$')
else:
    print('Aucun prix exploitable dans le sample meta (parsing à revoir).')

---

## Section 5 — Exploration libre

**Ce qu'on fait** : sortir des échantillons aléatoires pour les voir "à l'œil". Toutes les statistiques agrégées du monde ne remplacent pas le réflexe de regarder 10 lignes au hasard.

**Ce qu'on cherche** : du bizarre. Des champs qui ne ressemblent pas à ce que la doc HF promet. Des encodages cassés (mojibake), des HTML non échappés, des langues inattendues, des outliers énormes. Si tu trouves quelque chose d'intéressant, **note-le** dans `reports/audit_v1_amazon_reviews_2023.md` section 9 (Annexes) ou `BRAIN/learnings.md`.

In [ ]:
# Échantillon aléatoire reviews — 10 lignes pour observation à l'œil
reviews_df.select(
    'rating', 'title', 'text', 'parent_asin', 'verified_purchase',
    'helpful_vote', '_source_category',
).sample(n=10, seed=SEED).to_pandas()

In [ ]:
# Échantillon aléatoire meta — pour voir à quoi ressemblent les descriptions produit
meta_df.select(
    'main_category', 'title', 'price', 'description', 'parent_asin',
    '_source_category',
).sample(n=10, seed=SEED).to_pandas()

---

## Section 6 — Pour aller plus loin

Le notebook s'arrête ici délibérément. Les analyses **canoniques** (qualité full, distributions full, biais, leakages, sample queue-longue) sont dans les scripts officiels qui tournent en streaming sur les 348 M+ reviews :

- `make audit-quality` → `reports/audit_qualite_metrics.json` + figures NaN
- `make audit-dist` → `reports/audit_distributions_metrics.json` + figures distributions
- `make audit-bias` → `reports/audit_biais_leakage_metrics.json`
- `make audit` → enchaîne les trois (~9-12 min sur 15 cat)

**Synthèse humaine** : `reports/audit_v1_amazon_reviews_2023.md` (verdict + conditions à respecter en C06/C07)

**Méta-first amont** : `notebooks/00_meta_exploration.ipynb` (Range probes + filtre périmètre)

**Décisions tracées** : `BRAIN/decisions.md` (D-004 dataset, D-006 polars streaming, D-007 full vs sample, D-008 périmètre méta-first)

**Règles d'or à connaître** : `BRAIN/golden_rules.md` (R17 méta-first avant data-first, R18 biais-de-supposition-explicite)

Si tu utilises ce notebook pour creuser une question précise (ex : *"pourquoi y a-t-il autant de titres dupliqués ?"*), termine ta session par :

1. Un paragraphe de conclusion dans le notebook (cellule markdown finale).
2. Une remontée de la conclusion dans le rapport markdown ou dans `BRAIN/learnings.md`.
3. **Pas de `df.to_parquet(...)`** — l'audit ne modifie pas les données. Si tu veux corriger quelque chose, c'est en C06.